# The $(\alpha, K)$ difficulty grid

Two questions on one grid of mixtures, using the same draws for both.

**Is the separation condition plausible?** Assumption 3 asks that
$\eta = \gamma_{\mathrm{out}} - \gamma_{\mathrm{in}}$ be positive. At a finite horizon we
observe $\eta_n$, not $\eta$, and only through an estimate. Each mixture is therefore
classified from *simultaneous* confidence bounds on the entries of $\Gamma^{(n)}$:
**separated** when the lower bound is positive, **nonseparated** when the upper bound is
negative, **uncertain** in between. The third class is what the published Figure 2 could
not express: it estimated each $\Gamma_{k\ell}$ from two realisations at a single horizon
and read off the sign of a point estimate, so cells near the boundary reported the sign of
their own noise.

**Does clustering recover the partition, and does the data-driven rule find $K$?** Exact
recovery is the primary outcome, being what Theorems 3.5, 3.8 and 3.9 are statements about;
ARI is kept as a graded second reading of the same partition.

$\Gamma^{(n)}$ is estimated on an *independent* Monte Carlo sample, never on the
dissimilarity matrix the clustering ran on. Kernels are drawn once per `mixture_id` and held
fixed, so all four algorithms and both horizons see the same mixture and the comparison
between them is paired.

In [1]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clustering import (asw_select_k, exact_recovery, profile_distances,
                        profile_graph_k, profile_heights, safeguard_threshold)
from experiments import (ALGORITHMS, ResultsWriter, draw_markov_mixture,
                         estimate_gamma_paths, eta_rows, save_mixture, score_dataset,
                         stream, univariate_om, wilson_interval)
from figures import DIVERGING_CMAP, PAPER_STYLE, SEQUENTIAL_CMAP


## Setup

Everything the figures below report is produced by the sweep in this notebook: the mixtures,
the finite-horizon geometry, the four clustering algorithms and the two rules for $K$. The
knobs are all in the next cell, `N` first.

The sweep computes every unit the knobs ask for, every time: it never reads back rows left on
disk by an earlier run, and nothing it writes is ever picked up again. What the figures show is
what the cell above just ran. It writes its tables as it goes, so a run killed halfway leaves
what it had, but a rerun starts them over from scratch.

That costs about 9 s per mixture at $N = 200$, so roughly 4 h for the whole grid. Set $N$ to
something small -- 60 sequences over a couple of cells run in seconds -- to watch the section
rebuild itself. `SOURCE = "paper"` is the one way to skip the sweep: it reads the $N = 200$
tables the published section reports, the ones `sweep_recovery.py` and `sweep_khat.py` wrote
before the sweep moved in here.


In [2]:
# --- the knobs -------------------------------------------------------------
SOURCE   = "compute"     # "compute": run the sweep below at the knobs that follow.
                         # "paper":   skip it, read the N = 200 tables of the section.
N        = 800                                  # sequences per clustering dataset
R_MIX    = 30                                   # mixtures per (alpha, K) cell
ALPHAS   = [0.2, 0.3, 0.4, 0.5, 1.0, 5.0, 10.0]
KS       = list(range(2, 11))
HORIZONS = [1000]                          # nested prefixes of the same trajectories
HORIZON  = 1000                                 # the horizon the figures report
D_STATES = 5
COST     = "constant"
N_GAMMA  = 60            # independent trajectory pairs behind each entry of Gamma^(n)
LEVEL    = 0.95
SEED     = 20260901
PAM_RESTARTS = 1

ALGOS      = ["average", "pam"]                          # the main comparison
ALGOS_ALL  = ["single", "complete", "average", "pam"]    # everything the sweep scores
RULES      = ["safeguard", "asw-pam"]                    # the rule of the paper, and
                                                         # the one applied work uses
RULE_HORIZONS = [HORIZON]   # the rules are scored here only: the two ASW rules cost
                            # an order of magnitude more than everything else combined
K_MAX_ASW = 12              # the grid tops out at K = 10; 12 leaves ASW room to overshoot

RESULTS = Path("results")
FIGURES = Path("Figures/grid_800")
FIGURES.mkdir(parents=True, exist_ok=True)

# Where the sweep writes, and the tables the published section reports. The first are
# overwritten by every run and never read back; the second are read only under SOURCE =
# "paper", and never written to.
OUT   = {kind: RESULTS / f"grid_N{N}_{kind}.csv" for kind in ("cluster", "eta", "khat")}
PAPER = {"cluster": RESULTS / "recovery_cluster_main.csv",
         "eta":     RESULTS / "recovery_eta_main.csv",
         "khat":    RESULTS / "khat_grid_final.csv"}

om = univariate_om(COST, D_STATES, rng=stream(SEED, "cost", COST))
assumption = om.assumption_1()
metric_ok = all(v for k, v in assumption.items() if k.startswith("("))
print(f"cost scheme {om.name}: Assumption 1 holds = {metric_ok}, M = {om.M:.3f}")
if not metric_ok:
    print("  !! outside the metric framework -- report as robustness, not as confirmation")


cost scheme constant: Assumption 1 holds = True, M = 2.000


## The sweep

One `(alpha, K, mixture)` unit, end to end. The mixture is drawn once and held fixed, and
everything downstream is conditional on it: one dataset, one set of OM matrices, and on those
same matrices the four algorithms at the true $K$ and the two rules for $\hat K$. Scoring the
two blocks on one dataset is what makes the comparison between them paired, and it halves the
cost -- the OM matrices are the expensive part and were previously computed twice.

$\Gamma^{(n)}$ is the one thing not read off those matrices: it is estimated on an
*independent* Monte Carlo sample of trajectory pairs, never on the matrix the clustering ran
on. That is the two-level Monte Carlo the section rests on.


In [3]:
def evaluate_rules(D, rho, heights, n):
    """(rule, threshold, k_hat, labels) for the two rules for K, on one matrix.

    `safeguard` is the rule of the paper, max{sqrt(h_med h_max), h_max (log N / n)^(1/4)}
    over the single-linkage merge heights of the profile distances. `asw-pam` is what applied
    sequence analysis does, the default of WeightedCluster: it maximises the average
    silhouette width over a range of k, and is consistent for nothing. That is the comparison
    the paper makes -- a consistent rule against the one practice uses -- and not a search
    over heuristics, so nothing else is run.

    `safeguard` reads only rho, computed once for the matrix. `asw-pam` does not read rho at
    all: it clusters at every k and scores the partition, which is why it dominates the bill.
    """
    threshold = safeguard_threshold(rho, n, heights)
    k_hat, labels = profile_graph_k(None, rho=rho, threshold=threshold, return_labels=True)
    out = [("safeguard", threshold, k_hat, labels)]

    k_hat, labels = asw_select_k(D, k_max=K_MAX_ASW, method="pam",
                                 pam_restarts=PAM_RESTARTS, return_labels=True)
    out.append(("asw-pam", "", k_hat, labels))
    return out


def sweep_unit(alpha, K, m):
    """One mixture of one cell: (eta rows, clustering rows, K-selection rows)."""
    mixture = draw_markov_mixture(K, D_STATES, alpha, m, SEED)
    save_mixture(mixture, RESULTS / "mixtures")

    estimate = estimate_gamma_paths(mixture, om, HORIZONS, N_GAMMA,
                                    stream(SEED, "grid-gamma", alpha, K, m))
    geometry = eta_rows(mixture, om, estimate, level=LEVEL)

    rng = stream(SEED, "grid-data", alpha, K, m)
    X, truth = mixture.sample_dataset(N, max(HORIZONS), rng)
    matrices = om.matrices(X, HORIZONS)
    cluster = score_dataset(mixture, om, matrices, truth, N, HORIZONS, rng, 0,
                            algorithms=ALGORITHMS, pam_restarts=PAM_RESTARTS)

    selection = []
    for g, n in enumerate(HORIZONS):
        if n not in RULE_HORIZONS:
            continue
        rho = profile_distances(matrices[g])
        heights = profile_heights(rho)
        for rule, threshold, k_hat, labels in evaluate_rules(matrices[g], rho, heights, n):
            selection.append({
                "alpha": alpha, "K": K, "mixture_id": m, "dataset_id": 0, "n": int(n),
                "N": N, "d": mixture.d, "cost_scheme": om.name, "rule": rule,
                "threshold": threshold, "k_hat": int(k_hat),
                "k_correct": int(k_hat == K),
                "exact_recovery": int(exact_recovery(labels, truth)),
                "mixture_key": mixture.key,
            })
    return geometry, cluster, selection


FIELDS = {
    "cluster": ["alpha", "K", "d", "mixture_id", "n", "N", "cost_scheme", "algorithm",
                "dataset_id", "exact_recovery", "ari", "k_hat", "k_correct",
                "exact_recovery_at_k_hat", "pam_one_swap_certified", "pam_hit_cap",
                "mixture_key"],
    "eta": ["alpha", "K", "mixture_id", "n", "cost_scheme", "eta_hat", "eta_ci_low",
            "eta_ci_high", "separation_status", "n_pairs", "level", "mixture_key"],
    "khat": ["alpha", "K", "mixture_id", "dataset_id", "n", "N", "d", "cost_scheme",
             "rule", "threshold", "k_hat", "k_correct", "exact_recovery", "mixture_key"],
}


In [ ]:
units = [(a, K, m) for a in ALPHAS for K in KS for m in range(R_MIX)]

if SOURCE == "compute":
    print(f"{len(ALPHAS)} x {len(KS)} cells x {R_MIX} mixtures = {len(units)} units "
          f"at N = {N}, horizons {HORIZONS}")
    for path in OUT.values():           # every run starts its tables over
        path.unlink(missing_ok=True)
    writers = {kind: ResultsWriter(OUT[kind], FIELDS[kind]) for kind in FIELDS}
    start = time.perf_counter()
    try:
        for done, (alpha, K, m) in enumerate(units, start=1):
            geometry, cluster_rows, selection = sweep_unit(alpha, K, m)
            for kind, rows in (("eta", geometry), ("khat", selection),
                               ("cluster", cluster_rows)):
                writers[kind].write([{k: row.get(k, "") for k in FIELDS[kind]}
                                     for row in rows])
            if done % 10 == 0 or done == len(units):
                elapsed = time.perf_counter() - start
                left = (len(units) - done) * elapsed / done
                print(f"  {done:>5}/{len(units)}  alpha={alpha:<5} K={K:<2}  "
                      f"{elapsed/60:6.1f} min elapsed, ~{left/60:6.1f} min left", flush=True)
    finally:
        for writer in writers.values():
            writer.close()
    print(f"done in {(time.perf_counter() - start)/60:.1f} min")
else:
    print("SOURCE = 'paper': the sweep is not run, the N = 200 tables of the published "
          "section are read instead")


7 x 9 cells x 30 mixtures = 1890 units at N = 800, horizons [1000]
     10/1890  alpha=0.2   K=2      8.5 min elapsed, ~1604.4 min left
     20/1890  alpha=0.2   K=2     24.4 min elapsed, ~2277.6 min left
     30/1890  alpha=0.2   K=2     33.6 min elapsed, ~2082.6 min left


In [ ]:
def load(kind):
    """The rows of `kind`: what the sweep just wrote, or the tables of the section."""
    df = pd.read_csv(OUT[kind] if SOURCE == "compute" else PAPER[kind])
    return df[df.alpha.isin(ALPHAS) & df.K.isin(KS) & (df.mixture_id < R_MIX)]


cluster = load("cluster")
eta     = load("eta")
khat    = load("khat")

cluster = cluster[cluster.n == HORIZON]
eta     = eta[eta.n == HORIZON]
khat    = khat[khat.n == HORIZON]

per_cell = int(eta.groupby(["alpha", "K"]).mixture_id.nunique().max())
print(f"{len(ALPHAS)} x {len(KS)} cells, {per_cell} mixtures each, "
      f"N = {int(cluster.N.iloc[0])}, n = {HORIZON}")


## Reading a cell

Every number below is a Monte Carlo proportion over `R_MIX` mixtures, so every number
carries a Wilson interval. `grid` turns any per-mixture indicator into the array the
heatmaps draw, and `frame` is the axis furniture of the published figures.

In [ ]:
def grid(df, value, agg="mean"):
    """(len(ALPHAS), len(KS)) array of `value` aggregated per cell."""
    table = df.pivot_table(index="alpha", columns="K", values=value, aggfunc=agg)
    return table.reindex(index=ALPHAS, columns=KS).to_numpy(dtype=float)


def wilson_half_width(df, value):
    """Half-width of the Wilson interval on each cell's proportion, for the annotations."""
    out = np.full((len(ALPHAS), len(KS)), np.nan)
    for i, a in enumerate(ALPHAS):
        for j, K in enumerate(KS):
            s = df[(df.alpha == a) & (df.K == K)][value]
            if len(s):
                lo, hi = wilson_interval(int(s.sum()), len(s), LEVEL)
                out[i, j] = (hi - lo) / 2
    return out


def frame(ax, title):
    ax.set_xticks(range(len(KS)), [str(K) for K in KS])
    ax.set_yticks(range(len(ALPHAS)), [f"{a:g}" for a in ALPHAS])
    ax.set_xlabel(r"$K$")
    ax.set_ylabel(r"$\alpha$")
    ax.set_title(title, fontsize=9, pad=6)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.4)
        spine.set_color("0.7")


def heatmap(ax, P, annot=None, title="", cmap=SEQUENTIAL_CMAP, fmt="{:.2f}"):
    """Colour = the proportion; annotation = that proportion and, below it, `annot`."""
    im = ax.imshow(P, origin="lower", aspect="auto", cmap=cmap, vmin=0.0, vmax=1.0)
    frame(ax, title)
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            if not np.isfinite(P[i, j]):
                continue
            col = "white" if P[i, j] > 0.55 else "0.25"
            ax.text(j, i + 0.13, fmt.format(P[i, j]), ha="center", va="center",
                    fontsize=5.5, color=col)
            if annot is not None and np.isfinite(annot[i, j]):
                ax.text(j, i - 0.17, f"{annot[i, j]:+.2f}", ha="center", va="center",
                        fontsize=4.6, color=col, alpha=0.85)
    return im


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES / f"{name}.{ext}", bbox_inches="tight")
    print("figure written to", FIGURES / f"{name}.pdf")

## Plausibility of the separation condition

Colour is the **verdict balance**, $\Pr(\text{separated}) - \Pr(\text{nonseparated})$ at
level 0.95: blue where the interval establishes $\eta_n > 0$, red where it establishes
$\eta_n < 0$, and pale wherever it cannot decide. A pale cell is not a cell where the
condition fails; it is one where the horizon is too short to tell, and the published
Figure 2 had no way of saying so -- it read the sign of a point estimate and coloured the
cell as though the answer were known.

The big number is the separated share, the small one the median $\hat\eta_n$, so a cell at
0 still says how far it is.

In [ ]:
eta = eta.assign(
    is_separated=(eta.separation_status == "separated").astype(int),
    is_uncertain=(eta.separation_status == "uncertain").astype(int),
    is_nonseparated=(eta.separation_status == "nonseparated").astype(int),
)

P_sep = grid(eta, "is_separated")
P_non = grid(eta, "is_nonseparated")
balance = P_sep - P_non                 # +1 all separated, -1 all not, 0 no verdict
margin = grid(eta, "eta_hat", agg="median")

with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    im = ax.imshow(balance, origin="lower", aspect="auto", cmap=DIVERGING_CMAP,
                   vmin=-1.0, vmax=1.0)
    frame(ax, r"Assumption 3 at $n = %d$" % HORIZON)
    for i in range(balance.shape[0]):
        for j in range(balance.shape[1]):
            if not np.isfinite(balance[i, j]):
                continue
            col = "white" if abs(balance[i, j]) > 0.6 else "0.25"
            ax.text(j, i + 0.13, f"{P_sep[i, j]:.2f}", ha="center", va="center",
                    fontsize=5.5, color=col)
            ax.text(j, i - 0.17, f"{margin[i, j]:+.2f}", ha="center", va="center",
                    fontsize=4.6, color=col, alpha=0.85)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03,
                      ticks=[-1, -0.5, 0, 0.5, 1])
    cb.set_label(r"$\Pr(\eta_n > 0$ established$) - \Pr(\eta_n < 0$ established$)$",
                 fontsize=8)
    cb.ax.set_yticklabels(["all\nnot separated", "", "no\nverdict", "", "all\nseparated"],
                          fontsize=6)
    cb.outline.set_visible(False)
    fig.tight_layout()
    save(fig, "separation")
    plt.show()

print("median Wilson half-width on the separated proportion: "
      f"{np.nanmedian(wilson_half_width(eta, 'is_separated')):.3f}")

## Exact recovery at known $K$

Average linkage and PAM. Complete linkage is dropped: paired over the grid it is
indistinguishable from average (79 wins against 107, $p = 0.05$), so reporting both says
nothing. Single linkage is the estimator Theorem 3.5 is actually about, and is kept in the
appendix section below, where the published Figures 9 and 10 had it.

The linkages are cut at exactly the first $N-K$ merges rather than by a height threshold,
so every one of them obeys literally the same rule; PAM is the strictly-improving one-swap
algorithm of Theorem 3.8, and every run carries the flag certifying that its medoid set is
one-swap stationary.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0), sharey=True)
    for ax, algo in zip(axes.ravel(), ALGOS):
        sub = cluster[cluster.algorithm == algo]
        im = heatmap(ax, grid(sub, "exact_recovery"), grid(sub, "ari"), algo)
    cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
    cb.set_label("probability of exact recovery (annotated: mean ARI)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, "recovery")
    plt.show()

certified = cluster[cluster.algorithm == "pam"]
print(f"PAM certified one-swap stationary in "
      f"{int(certified.pam_one_swap_certified.sum())}/{len(certified)} runs; "
      f"hit the swap cap {int(certified.pam_hit_cap.sum())} times")

## The chain the section rests on

$(\alpha, K)$ is a *generative* difficulty knob, not the separation condition itself. What
governs recovery is $\eta_n$, and the grid is only a way of sweeping through it. Binning the
same mixtures by their estimated $\eta_n$ shows the chain directly, and the table is the whole
of it: curves drawn from these same numbers say no more than the ARI paths of the recovery
block already do.

In [ ]:
merged = cluster.merge(
    eta[["alpha", "K", "mixture_id", "eta_hat", "separation_status"]],
    on=["alpha", "K", "mixture_id"], how="left")

edges  = [-np.inf, 0.0, 0.02, 0.05, 0.10, 0.20, np.inf]
labels = [r"$\eta_n<0$", "0–.02", ".02–.05", ".05–.10", ".10–.20", r"$>$.20"]
merged["band"] = pd.cut(merged.eta_hat, edges, labels=labels)

rows = []
for band in labels:
    s = merged[merged.band == band]
    if not len(s):
        continue
    row = {"eta band": band, "mixtures": len(s) // len(ALGOS_ALL)}
    for a in ALGOS_ALL:
        row[a] = f"{100 * s[s.algorithm == a].exact_recovery.mean():.0f}%"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

## Selecting $K$ from the data

Two rules on the same matrices. `safeguard` is the rule of the paper,
$\max\{\sqrt{h_{\mathrm{med}}h_{\max}},\ h_{\max}(\log N/n)^{1/4}\}$ over the single-linkage
merge heights of the profile distances; `asw-pam` is what applied sequence analysis does,
maximising the average silhouette width over a range of $k$ -- the default of
`WeightedCluster`, and consistent for nothing.

Reported separately, as the two questions they are: does the rule find $K$, and does the
partition it implies recover $\mathcal P^\star$.

In [ ]:
TITLES = {"safeguard": r"safeguarded profile rule",
          "asw-pam":   r"maximal silhouette width (PAM)"}

with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, len(RULES), figsize=(4.8 * len(RULES), 3.6), sharey=True)
    for ax, rule in zip(axes, RULES):
        sub = khat[khat.rule == rule]
        im = heatmap(ax, grid(sub, "k_correct"), grid(sub, "exact_recovery"), TITLES[rule])
    cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    cb.set_label(r"$\Pr(\hat K = K)$ (annotated: $\Pr$ exact partition)", fontsize=8)
    cb.outline.set_visible(False)
    save(fig, "k_selection")
    plt.show()

summary = []
for rule in RULES:
    s = khat[khat.rule == rule]
    lo, hi = wilson_interval(int(s.k_correct.sum()), len(s), LEVEL)
    summary.append({"rule": rule, "K_hat = K": f"{100 * s.k_correct.mean():.1f}%",
                    "95% CI": f"[{100*lo:.0f}, {100*hi:.0f}]",
                    "exact partition": f"{100 * s.exact_recovery.mean():.1f}%"})
print(pd.DataFrame(summary).to_string(index=False))

## The price of not knowing $K$

Cutting at $\hat K$ rather than at the true $K$ costs the difference between the two columns
below. It is not the same as the error rate of the rule: a wrong $\hat K$ can still leave
most of the partition intact, and a right $\hat K$ does not guarantee the partition.

In [ ]:
rows = []
for algo in ALGOS_ALL:
    s = cluster[cluster.algorithm == algo]
    rows.append({"algorithm": algo,
                 "known K": f"{100 * s.exact_recovery.mean():.1f}%",
                 "at K_hat": f"{100 * s.exact_recovery_at_k_hat.mean():.1f}%",
                 "K_hat correct": f"{100 * s.k_correct.mean():.1f}%"})
print(pd.DataFrame(rows).to_string(index=False))

## Appendix: single linkage

The estimator Theorem 3.5 is about. It is the weakest of the three in practice -- single
linkage merges two blocks on the *smallest* dissimilarity between them, so one aberrant
sequence chains them together and the cut at $K$ then spends a whole block on that sequence.
Average linkage carries the main figure for that reason; the theory covers both, and every
bracketed linkage between them, by Remark 3.4.

In [ ]:
with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    sub = cluster[cluster.algorithm == "single"]
    im = heatmap(ax, grid(sub, "exact_recovery"), grid(sub, "ari"), "single linkage")
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("probability of exact recovery (annotated: mean ARI)", fontsize=8)
    cb.outline.set_visible(False)
    fig.tight_layout()
    save(fig, "recovery_single_linkage")
    plt.show()